# 🏙️ Chulalongkorn Smart Campus Energy Intelligence
## Foundation-Model-Based Energy Forecasting & Multi-Agent Optimization for Smart Building Energy Management

**Dataset:** CU-BEMS — Chulalongkorn University Building Energy Management System (Smart Building Energy & IAQ Data), Bangkok, Thailand · 7 floors · 33 zones · 1-minute resolution · Jul 2018 – Dec 2019.

---

### Abstract
This notebook turns the CU-BEMS smart-building dataset into a **reproducible electricity-forecasting pipeline** and a **modular multi-agent decision system**. We forecast building electricity demand at multiple horizons (1 hour and 24 hours ahead) for the whole building and per load type (AC / lighting / plug), using a **strict chronological train/validation/test split** and **leakage-safe lag/rolling features**. We benchmark a ladder of models — persistence and seasonal-naïve baselines, linear/Ridge regression, gradient-boosted trees — and **optionally** a time-series **foundation model (Amazon Chronos-Bolt, zero-shot)** when the environment allows it. Forecasts then drive three downstream agents: a **forecast-residual anomaly detector** (benchmarked against Isolation Forest), a **comfort monitor**, and a **forecast-informed scheduler** that proposes energy-saving actions only when comfort constraints are respected. A **coordinator agent** combines and ranks the recommendations.

> **Honesty notes built into this notebook**
> - All savings/CO₂ figures use **explicit kW → kWh conversion** (`energy_kWh = mean_power_kW × interval_hours`) and clearly-stated tariff/emission assumptions. They are *illustrative*, not validated bills.
> - Foundation models are **optional**. The notebook runs end-to-end **offline**, with no internet, no GPU, and no extra input files. If foundation-model packages/weights are unavailable, that section is skipped automatically.
> - We only claim a model is "better" where the **metrics on the held-out test set** show it.

## 0 · Problem statement, targets & horizons

**Problem.** Campus buildings are operated reactively (static schedules, siloed control). Given 18 months of sub-metered electricity and indoor-environment data from one Chulalongkorn University building, can we (a) **forecast** electricity demand accurately, (b) **detect** abnormal consumption, and (c) **recommend** energy-saving schedule changes that do **not** violate occupant comfort?

**Forecasting targets** (all are *average electrical power in kW* over the interval):
| Target | Meaning |
|---|---|
| `Bldg_Total_kW` | Whole-building electricity demand |
| `Bldg_AC_total_kW` | Air-conditioning (HVAC) load |
| `Bldg_Light_total_kW` | Lighting load |
| `Bldg_Plug_total_kW` | Plug / equipment load |
| `F{n}_Total_kW` | Floor-level demand (data-quality permitting) |

**Horizons:** **h = 1 hour** (operational) and **h = 24 hours** (day-ahead). Day-ahead **peak-demand error** is reported separately.

**Evaluation protocol (leakage-safe):** chronological split → fit/scale on train only → select on validation → report once on test. Features use **only past information**; targets are `y(t+h)`.

In [ ]:
# ============================================================
# CONFIG & GUARDED IMPORTS  — Kaggle-safe, offline-first
# ============================================================
import os, gc, re, warnings, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Execution switches ───────────────────────────────────────
# Foundation model is ON by default here. It is still fully GUARDED: if internet /
# packages / weights are unavailable, that section prints why and skips — the
# notebook never crashes. Flip to False to force the offline baseline-only run.
USE_FOUNDATION_MODEL = True         # master switch for the Chronos section
MODEL_BACKEND        = "auto"       # "auto" -> use whatever ML libs are installed
CHRONOS_MODEL        = "amazon/chronos-bolt-small"  # small by default (CPU-friendly)
CHRONOS_MAX_WINDOWS  = 400          # cap rolling-origin evals to bound runtime

# ── Modeling frequency & horizons ────────────────────────────
FREQ          = "1h"        # forecasting resolution
INTERVAL_H    = 1.0         # hours per step at FREQ  (energy_kWh = power_kW * INTERVAL_H)
SEASONAL_M    = 24          # daily seasonality in steps (24 hours)
HORIZONS      = [1, 24]     # forecast horizons in steps
SPLIT_FRACS   = (0.70, 0.15, 0.15)  # train / val / test (chronological)

# ── Economic / environmental assumptions (ILLUSTRATIVE — edit freely) ──
# Thailand context. These are transparent assumptions, not validated bills.
ELEC_TARIFF_THB_PER_KWH = 4.5     # ~MEA/PEA large-building tariff order of magnitude
GRID_CO2_KG_PER_KWH     = 0.50    # ~Thailand grid emission factor (approx.)

# ── Optional ML libraries: detect, never require ─────────────
OPTIONAL_LIBS = {}
def _try(name, importer):
    try:
        OPTIONAL_LIBS[name] = importer()
        return True
    except Exception as e:
        OPTIONAL_LIBS[name] = None
        print(f"   • {name:14s} not available ({type(e).__name__}) — will skip / fall back")
        return False

print("🔎 Detecting optional ML libraries (all are optional):")
_try("lightgbm",  lambda: __import__("lightgbm"))
_try("xgboost",   lambda: __import__("xgboost"))
HAS_LGBM = OPTIONAL_LIBS["lightgbm"] is not None
HAS_XGB  = OPTIONAL_LIBS["xgboost"]  is not None

# ── Plot identity (dark, publication-style) ──────────────────
COLORS = {
    "energy": "#FF6B35", "comfort": "#00B4D8", "maintain": "#2EC4B6",
    "coord": "#9B5DE5", "accent1": "#F15BB5", "accent2": "#FEE440",
    "ok": "#06D6A0", "bad": "#EF476F",
    "dark": "#1A1A2E", "light": "#E8E8E8", "bg": "#0F0F1A", "grid": "#2A2A3E",
}
plt.rcParams.update({
    "figure.facecolor": COLORS["bg"], "axes.facecolor": COLORS["bg"],
    "axes.edgecolor": COLORS["grid"], "axes.labelcolor": "#FFFFFF",
    "text.color": "#FFFFFF", "xtick.color": "#AAAAAA", "ytick.color": "#AAAAAA",
    "grid.color": COLORS["grid"], "grid.alpha": 0.3, "font.size": 11,
    "axes.titlesize": 13, "axes.labelsize": 12, "figure.dpi": 110,
    "figure.figsize": (14, 6),
})

print(f"\n✅ Config ready | pandas {pd.__version__} | numpy {np.__version__}")
print(f"   FREQ={FREQ}  horizons={HORIZONS}  split={SPLIT_FRACS}")
print(f"   USE_FOUNDATION_MODEL={USE_FOUNDATION_MODEL}  LightGBM={HAS_LGBM}  XGBoost={HAS_XGB}")

## 1 · Robust data loading (Kaggle-safe path auto-detection)

The original notebook hard-coded one `/kaggle/input/...` path. Here we keep that as the **first guess**, then **auto-detect** the dataset folder by searching `/kaggle/input` recursively for the canonical `2018Floor1.csv` / `2019Floor1.csv` files. If nothing is found we raise a **clear, actionable error** naming the dataset to attach. No new mandatory inputs are introduced.

In [ ]:
# ============================================================
# DATA DIR RESOLUTION — try hard-coded path, else search /kaggle/input
# ============================================================
PRIMARY_DATA_DIR = Path("/kaggle/input/cubems-smart-building-energy-and-iaq-data")
# A few extra known spellings of this Kaggle dataset slug:
EXTRA_GUESSES = [
    Path("/kaggle/input/datasets/claytonmiller/cubems-smart-building-energy-and-iaq-data"),
    Path("/kaggle/input/cu-bems"),
]
INPUT_SEARCH_ROOTS = [Path("/kaggle/input")]
CANONICAL_FILES = ["2018Floor1.csv", "2019Floor1.csv"]

def _looks_like_cubems(d: Path) -> bool:
    try:
        return any((d / f).exists() for f in CANONICAL_FILES)
    except Exception:
        return False

def resolve_data_dir(primary, extra_guesses, search_roots):
    # 1) direct guesses
    for cand in [primary, *extra_guesses]:
        if cand and _looks_like_cubems(cand):
            return cand
    # 2) recursive search under the Kaggle input roots
    for root in search_roots:
        if not root.exists():
            continue
        for hit in root.rglob("2018Floor1.csv"):
            if _looks_like_cubems(hit.parent):
                return hit.parent
    # 3) clear, actionable failure
    raise FileNotFoundError(
        "CU-BEMS data not found.\n"
        "Attach the Kaggle dataset 'CU-BEMS Smart Building Energy and IAQ Data' "
        "(files like 2018Floor1.csv … 2019Floor7.csv) to this notebook.\n"
        f"Searched: {[str(primary)] + [str(g) for g in extra_guesses]} and rglob under {search_roots}."
    )

DATA_DIR = resolve_data_dir(PRIMARY_DATA_DIR, EXTRA_GUESSES, INPUT_SEARCH_ROOTS)
csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"📂 Using DATA_DIR = {DATA_DIR}")
print(f"   found {len(csv_files)} CSV files; "
      f"first few: {[f.name for f in csv_files[:4]]}")
assert any(_looks_like_cubems(DATA_DIR) for _ in [0]), "Resolved folder missing canonical files."

In [ ]:
# ============================================================
# COLUMN PARSER + FLOOR LOADER (1-min CSV -> 15-min mean power)
# ============================================================
def parse_column(col_name):
    # Parse a CU-BEMS column name into structured metadata, or None.
    if col_name == "Date":
        return None
    m = re.match(r"z(\d+)_", col_name)
    if not m:
        return None
    zone = int(m.group(1))
    rem = col_name[m.end():]
    if "AC" in rem:
        ctype, um = "ac", re.match(r"(AC\d+)", rem)
        unit_id = um.group(1) if um else "AC"
    elif "Light" in rem and "kW" in rem:
        ctype, unit_id = "lighting", "Light"
    elif "Plug" in rem:
        ctype, unit_id = "plug", "Plug"
    elif "degC" in rem:
        ctype, unit_id = "temperature", (re.match(r"(S\d+)", rem) or re.match(r"()", rem)).group(1) or "S"
    elif "RH%" in rem:
        ctype, unit_id = "humidity", (re.match(r"(S\d+)", rem) or re.match(r"()", rem)).group(1) or "S"
    elif "lux" in rem:
        ctype, unit_id = "lux", (re.match(r"(S\d+)", rem) or re.match(r"()", rem)).group(1) or "S"
    else:
        ctype, unit_id = "unknown", rem
    return {"zone": zone, "type": ctype, "unit_id": unit_id, "original": col_name}

RESAMPLE_RAW = "15min"   # working resolution for the raw per-channel data

def load_floor(floor_num, resample=RESAMPLE_RAW):
    # Load + combine 2018 & 2019 for one floor; coerce numerics; resample to mean power (kW).
    frames = []
    for year in (2018, 2019):
        fpath = DATA_DIR / f"{year}Floor{floor_num}.csv"
        if fpath.exists():
            df = pd.read_csv(fpath)
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            n_bad = int(df["Date"].isna().sum())
            if n_bad:
                print(f"    ⚠️ {year}Floor{floor_num}: dropped {n_bad} unparseable timestamps")
            df = df.dropna(subset=["Date"])
            frames.append(df)
    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True).sort_values("Date").set_index("Date")
    for c in combined.columns:
        combined[c] = pd.to_numeric(combined[c], errors="coerce")
    combined = combined[~combined.index.duplicated(keep="first")]
    if resample:
        combined = combined.resample(resample).mean()   # mean power over each bin (kW)
    return combined

print("⏳ Loading 7 floors (1-min → 15-min mean power)...\n")
floors, column_registry = {}, []
for f_num in range(1, 8):
    df = load_floor(f_num)
    if df is None:
        print(f"  ⚠️ Floor {f_num}: no files found, skipping")
        continue
    floors[f_num] = df
    for col in df.columns:
        meta = parse_column(col)
        if meta:
            meta["floor"] = f_num
            column_registry.append(meta)
    print(f"  ✅ Floor {f_num}: {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols | "
          f"{df.index.min():%Y-%m-%d} → {df.index.max():%Y-%m-%d}")
col_registry_df = pd.DataFrame(column_registry)
print(f"\n📋 {len(col_registry_df)} data channels across {len(floors)} floors")
gc.collect()

## 2 · Data-quality audit

Before modeling we map the reliability of every channel. In a real campus
deployment an agent must know **which streams it can trust** — a missing
temperature reading is a control blind spot, not just a gap. Energy sub-meters
are typically reliable; environmental sensors have far more missing data.

In [ ]:
# ============================================================
# DATA-QUALITY AUDIT — missing %, zero %, basic stats per channel
# ============================================================
TYPE_LABELS = {"ac": "AC", "lighting": "Lighting", "plug": "Plug",
               "temperature": "Temp", "humidity": "Humidity", "lux": "Lux"}

q_records = []
for f_num, df in floors.items():
    n = len(df)
    for col in df.columns:
        meta = parse_column(col)
        if not meta:
            continue
        s = df[col]
        q_records.append({
            "floor": f_num, "zone": meta["zone"], "type": meta["type"],
            "column": col, "missing_pct": float(s.isna().mean() * 100),
            "zero_pct": float((s == 0).mean() * 100),
            "mean": float(s.mean()), "min": float(s.min()), "max": float(s.max()),
        })
quality_df = pd.DataFrame(q_records)

# Per-type summary -> reused by the DataQualityAgent later
data_quality_report = {}
print("📊 DATA-QUALITY SUMMARY (per channel type)")
print("="*64)
print(f"{'Type':<10}{'#chan':>6}{'avg miss%':>11}{'max miss%':>11}{'avg zero%':>11}")
print("-"*64)
for t in ["ac", "lighting", "plug", "temperature", "humidity", "lux"]:
    sub = quality_df[quality_df["type"] == t]
    if sub.empty:
        continue
    rec = {"n_channels": int(len(sub)),
           "avg_missing_pct": float(sub["missing_pct"].mean()),
           "max_missing_pct": float(sub["missing_pct"].max()),
           "avg_zero_pct": float(sub["zero_pct"].mean())}
    data_quality_report[t] = rec
    print(f"{TYPE_LABELS[t]:<10}{rec['n_channels']:>6}{rec['avg_missing_pct']:>11.2f}"
          f"{rec['max_missing_pct']:>11.2f}{rec['avg_zero_pct']:>11.2f}")
overall_missing = float(quality_df["missing_pct"].mean())
print("-"*64)
print(f"Overall missing rate across {len(quality_df)} channels: {overall_missing:.2f}%")
print("\nReading: AC zeros are mostly 'unit off' (not failures). Environmental "
      "sensors carry the most missing data — handled carefully downstream.")

# Compact visual: average missing % by type
fig, ax = plt.subplots(figsize=(9, 4))
order = [t for t in ["ac","lighting","plug","temperature","humidity","lux"] if t in data_quality_report]
vals = [data_quality_report[t]["avg_missing_pct"] for t in order]
ax.barh([TYPE_LABELS[t] for t in order][::-1], vals[::-1], color=COLORS["comfort"], alpha=0.9)
for i, v in enumerate(vals[::-1]):
    ax.text(v + 0.2, i, f"{v:.1f}%", va="center", color="white", fontweight="bold")
ax.set_xlabel("Average missing %"); ax.set_title("Missing data by channel type")
plt.tight_layout(); plt.show()

## 3 · Building & floor aggregates (units made explicit)

Each channel is **average electrical power in kW** over its time bin. Summing
several channels **at the same timestamp** is valid (instantaneous powers add):
`kW + kW → kW`. We build floor- and building-level **power** series with an
explicit `_kW` suffix so units are never ambiguous, plus floor/building
**environmental averages** (°C, %RH, lux). Converting power to **energy (kWh)**
is done separately in the next section.

In [ ]:
# ============================================================
# AGGREGATES — floor & building POWER (kW) + environment averages
# ============================================================
def cols_of(df, ctype):
    return [c for c in df.columns if (parse_column(c) or {}).get("type") == ctype]

floor_frames = {}
for f_num, df in floors.items():
    a = pd.DataFrame(index=df.index)
    ac, li, pl = cols_of(df, "ac"), cols_of(df, "lighting"), cols_of(df, "plug")
    te, hu, lx = cols_of(df, "temperature"), cols_of(df, "humidity"), cols_of(df, "lux")
    # sum of simultaneous powers = total power (kW). min_count=1 -> all-NaN stays NaN.
    a[f"F{f_num}_AC_total_kW"]    = df[ac].sum(axis=1, min_count=1) if ac else 0.0
    a[f"F{f_num}_Light_total_kW"] = df[li].sum(axis=1, min_count=1) if li else 0.0
    a[f"F{f_num}_Plug_total_kW"]  = df[pl].sum(axis=1, min_count=1) if pl else 0.0
    a[f"F{f_num}_Total_kW"] = a[[f"F{f_num}_AC_total_kW", f"F{f_num}_Light_total_kW",
                                 f"F{f_num}_Plug_total_kW"]].sum(axis=1, min_count=1)
    if te: a[f"F{f_num}_Temp_avg"] = df[te].mean(axis=1)
    if hu: a[f"F{f_num}_RH_avg"]   = df[hu].mean(axis=1)
    if lx: a[f"F{f_num}_Lux_avg"]  = df[lx].mean(axis=1)
    floor_frames[f_num] = a

building = pd.concat(floor_frames.values(), axis=1).sort_index()

# Building POWER totals (kW)
present = list(floors.keys())
building["Bldg_AC_total_kW"]    = building[[f"F{i}_AC_total_kW"    for i in present]].sum(axis=1, min_count=1)
building["Bldg_Light_total_kW"] = building[[f"F{i}_Light_total_kW" for i in present]].sum(axis=1, min_count=1)
building["Bldg_Plug_total_kW"]  = building[[f"F{i}_Plug_total_kW"  for i in present]].sum(axis=1, min_count=1)
building["Bldg_Total_kW"]       = building[["Bldg_AC_total_kW","Bldg_Light_total_kW",
                                            "Bldg_Plug_total_kW"]].sum(axis=1, min_count=1)
# Building ENVIRONMENT averages (only floors that have sensors)
for src, dst in [("Temp_avg","Bldg_Temp_avg"), ("RH_avg","Bldg_RH_avg"), ("Lux_avg","Bldg_Lux_avg")]:
    cc = [f"F{i}_{src}" for i in present if f"F{i}_{src}" in building.columns]
    building[dst] = building[cc].mean(axis=1) if cc else np.nan

def add_calendar(df):
    out = df.copy()
    idx = out.index
    out["hour"] = idx.hour.to_numpy()
    out["dow"] = idx.dayofweek.to_numpy()
    out["month"] = idx.month.to_numpy()
    out["is_weekend"] = (idx.dayofweek.to_numpy() >= 5).astype(int)
    out["is_working_hours"] = (((idx.hour.to_numpy() >= 8) & (idx.hour.to_numpy() < 18))
                               & (idx.dayofweek.to_numpy() < 5)).astype(int)
    return out

print(f"🏢 Building master frame: {building.shape[0]:,} rows × {building.shape[1]} cols "
      f"@ {RESAMPLE_RAW}")
print(f"   Coverage: {building.index.min():%Y-%m-%d} → {building.index.max():%Y-%m-%d}")
print(f"   Peak building power : {building['Bldg_Total_kW'].max():.1f} kW")
print(f"   Mean building power : {building['Bldg_Total_kW'].mean():.1f} kW")
if building['Bldg_Temp_avg'].notna().any():
    print(f"   Mean indoor temp    : {building['Bldg_Temp_avg'].mean():.1f} °C")
    print(f"   Mean indoor RH      : {building['Bldg_RH_avg'].mean():.1f} %")

## 4 · kW vs kWh — the unit rule, stated once and used everywhere

The channels are **average power (kW)** over each interval. **Energy** is the
integral of power over time:

$$\text{energy\_kWh} = \overline{\text{power}}_{\text{kW}} \times \text{interval\_hours}$$

- 15-min bins → `interval_hours = 0.25`
- 1-hour bins → `interval_hours = 1.0`

So total energy over a series = **(sum of per-bin mean powers) × interval_hours**.
We **never** add kW values and call the result kWh. All cost (THB) and CO₂
figures below flow through this single helper. Tariff and grid-emission factors
are **transparent assumptions** (Thailand context), not validated bills.

In [ ]:
# ============================================================
# ENERGY (kWh) HELPERS + correct totals/cost/CO2 + energy signature
# ============================================================
def series_energy_kwh(power_kw_series, interval_hours):
    # total kWh = sum(mean power per bin) * hours-per-bin ; NaNs ignored
    return float(np.nansum(np.asarray(power_kw_series, dtype=float)) * interval_hours)

def energy_cost_thb(kwh):    return kwh * ELEC_TARIFF_THB_PER_KWH
def energy_co2_kg(kwh):      return kwh * GRID_CO2_KG_PER_KWH

RAW_INTERVAL_H = 0.25  # 15-min bins
span_days = (building.index.max() - building.index.min()).total_seconds() / 86400.0
tot_kwh   = series_energy_kwh(building["Bldg_Total_kW"], RAW_INTERVAL_H)
ac_kwh    = series_energy_kwh(building["Bldg_AC_total_kW"], RAW_INTERVAL_H)
daily_kwh = tot_kwh / max(span_days, 1e-9)

print("⚡ ENERGY ACCOUNTING (energy_kWh = mean_power_kW × interval_hours, interval=0.25 h)")
print("="*68)
print(f"   Observed span        : {span_days:.0f} days")
print(f"   Total electricity    : {tot_kwh/1000:,.1f} MWh   (AC share {ac_kwh/tot_kwh*100:.1f}%)")
print(f"   Avg consumption      : {daily_kwh:,.0f} kWh/day")
print(f"   Illustrative cost    : ฿{energy_cost_thb(daily_kwh):,.0f}/day  "
      f"(@ ฿{ELEC_TARIFF_THB_PER_KWH}/kWh, assumption)")
print(f"   Illustrative CO2     : {energy_co2_kg(daily_kwh):,.0f} kgCO2/day "
      f"(@ {GRID_CO2_KG_PER_KWH} kg/kWh, assumption)")

# ---- Energy signature (one compact figure) ----
bnum = building.select_dtypes("number")
bcal = add_calendar(bnum)
fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.25, top=0.92, bottom=0.08, left=0.07, right=0.97)
fig.suptitle("Energy signature — Chulalongkorn campus building (CU-BEMS)", fontsize=16, fontweight="bold")

ax1 = fig.add_subplot(gs[0, :])
monthly = bnum[["Bldg_AC_total_kW","Bldg_Light_total_kW","Bldg_Plug_total_kW"]].resample("ME").mean()
ax1.plot(monthly.index, monthly["Bldg_AC_total_kW"], "-o", color=COLORS["energy"], label="AC")
ax1.plot(monthly.index, monthly["Bldg_Light_total_kW"], "-s", color=COLORS["accent2"], label="Lighting")
ax1.plot(monthly.index, monthly["Bldg_Plug_total_kW"], "-^", color=COLORS["accent1"], label="Plug")
ax1.set_ylabel("Mean power (kW)"); ax1.set_title("Monthly mean power by load type")
ax1.legend(framealpha=0.3); ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

ax2 = fig.add_subplot(gs[1, 0])
wd = bcal[bcal["is_weekend"] == 0].groupby("hour")["Bldg_Total_kW"].mean()
we = bcal[bcal["is_weekend"] == 1].groupby("hour")["Bldg_Total_kW"].mean()
ax2.plot(wd.index, wd.values, "-o", color=COLORS["energy"], label="Weekday")
ax2.plot(we.index, we.values, "-s", color=COLORS["comfort"], label="Weekend")
ax2.axvspan(8, 18, alpha=0.08, color="white"); ax2.set_xlabel("Hour"); ax2.set_ylabel("Mean power (kW)")
ax2.set_title("Daily load curve"); ax2.legend(framealpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
mix = [bnum[c].mean() for c in ["Bldg_AC_total_kW","Bldg_Light_total_kW","Bldg_Plug_total_kW"]]
ax3.pie(mix, labels=[f"AC\n{mix[0]:.0f} kW", f"Light\n{mix[1]:.0f} kW", f"Plug\n{mix[2]:.0f} kW"],
        colors=[COLORS["energy"], COLORS["accent2"], COLORS["accent1"]],
        autopct="%1.0f%%", wedgeprops=dict(width=0.5, edgecolor=COLORS["bg"]),
        textprops={"color": "white"})
ax3.set_title("Average load mix")
plt.show()

## 5 · Indoor environment & comfort baseline (Bangkok office/classroom context)

The original notebook applied **NHS hospital** thresholds (operating-theatre /
ward bands). That is the wrong context. CU-BEMS is a **university building in
Bangkok** (offices, classrooms, labs). We use **cooling-season office comfort**
bands appropriate to a hot-humid climate, consistent with **ASHRAE 55** and
common Thai practice — and we flag them as **approximate**:

- Temperature comfort: **22–27 °C** (cooling season)
- Relative humidity:   **30–70 %RH** (Bangkok is humid; 60% is often exceeded)

These bands feed the **ComfortAgent** later; they are not safety-critical clinical limits.

In [ ]:
# ============================================================
# COMFORT BASELINE — campus office/classroom bands (approximate)
# ============================================================
COMFORT = {
    "temp_low": 22.0, "temp_high": 27.0,   # deg C, cooling-season office (ASHRAE 55-ish)
    "rh_low": 30.0,  "rh_high": 70.0,      # %RH, hot-humid Bangkok context
}

# Per zone-sensor violation rates (used by ComfortAgent)
comfort_rows = []
for f_num, df in floors.items():
    for col in cols_of(df, "temperature"):
        s = df[col].dropna()
        if len(s) == 0:
            continue
        z = parse_column(col)["zone"]
        comfort_rows.append({
            "label": f"F{f_num}Z{z}", "floor": f_num, "zone": z,
            "too_hot_pct": float((s > COMFORT["temp_high"]).mean()*100),
            "too_cold_pct": float((s < COMFORT["temp_low"]).mean()*100),
            "mean_temp": float(s.mean()),
        })
comfort_df = pd.DataFrame(comfort_rows)
comfort_df["violation_pct"] = comfort_df["too_hot_pct"] + comfort_df["too_cold_pct"]

if not comfort_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    cd = comfort_df.sort_values("violation_pct")
    axes[0].barh(cd["label"], cd["too_hot_pct"], color=COLORS["bad"], label=f">{COMFORT['temp_high']}°C")
    axes[0].barh(cd["label"], -cd["too_cold_pct"], color=COLORS["comfort"], label=f"<{COMFORT['temp_low']}°C")
    axes[0].axvline(0, color="white", lw=0.6); axes[0].set_xlabel("% time in violation")
    axes[0].set_title("Thermal comfort violations by zone"); axes[0].legend(framealpha=0.3, fontsize=8)

    if building["Bldg_Temp_avg"].notna().any():
        bh = add_calendar(building.select_dtypes("number").resample("1h").mean())
        t_wd = bh[bh["is_weekend"] == 0].groupby("hour")["Bldg_Temp_avg"].mean()
        axes[1].plot(t_wd.index, t_wd.values, "-o", color=COLORS["energy"], label="Weekday mean")
        axes[1].axhspan(COMFORT["temp_low"], COMFORT["temp_high"], color=COLORS["ok"], alpha=0.12,
                        label="Comfort band")
        axes[1].set_xlabel("Hour"); axes[1].set_ylabel("Indoor temp (°C)")
        axes[1].set_title("Daily indoor-temperature profile"); axes[1].legend(framealpha=0.3, fontsize=8)
    plt.tight_layout(); plt.show()
    worst = comfort_df.nlargest(3, "violation_pct")
    print("Worst-served zones (highest % time outside comfort band):")
    for _, r in worst.iterrows():
        print(f"   {r['label']}: {r['violation_pct']:.1f}% "
              f"(hot {r['too_hot_pct']:.1f}%, cold {r['too_cold_pct']:.1f}%, mean {r['mean_temp']:.1f}°C)")

## 6 · Forecasting dataset, leakage-safe features & chronological split

We resample the building/floor **power** series to **hourly** mean power and
forecast each target at horizons **h = 1** and **h = 24**.

**Leakage safety — the rules we enforce:**
1. Every feature at forecast origin *t* uses **only data up to and including *t***
   (lags, rolling stats, calendar). The target is `y(t+h)`, strictly in the future.
2. Small gaps are forward-filled with a **short past-only limit** (≤ 3 h); larger
   gaps are left as `NaN` and **dropped** from the supervised set (never interpolated
   with future values).
3. The split is **chronological**: the **test window is the final calendar slice**
   and is **identical for every model, target and horizon**. Scalers/models are fit
   on **train only**; **validation** is used for selection; **test** is touched once.

In [ ]:
# ============================================================
# HOURLY MODELING FRAME + TARGETS + leakage-safe feature builder
# ============================================================
model_df = building.select_dtypes("number").resample(FREQ).mean()   # hourly mean power (kW)

present = sorted(floors.keys())
TARGETS = ["Bldg_Total_kW", "Bldg_AC_total_kW", "Bldg_Light_total_kW", "Bldg_Plug_total_kW"]
FLOOR_TARGETS = [f"F{i}_Total_kW" for i in present if f"F{i}_Total_kW" in model_df.columns]
PRIMARY_TARGET = "Bldg_Total_kW"

LAGS  = [1, 2, 3, 24, 48, 168]      # 1h,2h,3h, 1d,2d, 7d
ROLLS = [3, 24]                      # rolling windows (hours)

def build_features(y):
    # All features use information up to and including origin t (no look-ahead).
    f = pd.DataFrame(index=y.index)
    f["lag_0"] = y.values                                  # current level (known at t)
    for L in LAGS:
        f[f"lag_{L}"] = y.shift(L).values
    f["roll_mean_3"]  = y.rolling(3,  min_periods=3).mean().values
    f["roll_mean_24"] = y.rolling(24, min_periods=24).mean().values
    f["roll_std_24"]  = y.rolling(24, min_periods=24).std().values
    f["roll_min_24"]  = y.rolling(24, min_periods=24).min().values
    f["roll_max_24"]  = y.rolling(24, min_periods=24).max().values
    idx = y.index
    H = idx.hour.to_numpy(); D = idx.dayofweek.to_numpy()
    f["hour"] = H; f["dow"] = D; f["month"] = idx.month.to_numpy()
    f["is_weekend"] = (D >= 5).astype(int)
    f["is_working_hours"] = (((H >= 8) & (H < 18)) & (D < 5)).astype(int)
    f["sin_hour"] = np.sin(2*np.pi*H/24); f["cos_hour"] = np.cos(2*np.pi*H/24)
    f["sin_dow"]  = np.sin(2*np.pi*D/7);  f["cos_dow"]  = np.cos(2*np.pi*D/7)
    return f

FEATURE_COLS = list(build_features(model_df[PRIMARY_TARGET]).columns)
print(f"🧱 Modeling frame: {model_df.shape[0]:,} hourly rows  ({FREQ})")
print(f"   Targets        : {TARGETS}")
print(f"   Floor targets  : {FLOOR_TARGETS}")
print(f"   #features      : {len(FEATURE_COLS)}  (lags {LAGS}, rolls {ROLLS}, calendar+cyclic)")

In [ ]:
# ============================================================
# CHRONOLOGICAL SPLIT (70/15/15) + supervised-set builder + MASE scale
# ============================================================
full_idx = model_df.index
n = len(full_idx)
i_tr, i_va = int(n*SPLIT_FRACS[0]), int(n*(SPLIT_FRACS[0]+SPLIT_FRACS[1]))
TRAIN_END = full_idx[i_tr-1]
VAL_END   = full_idx[i_va-1]
print(f"🗂  Chronological split (no shuffling):")
print(f"   TRAIN : {full_idx[0]:%Y-%m-%d %H:%M} → {TRAIN_END:%Y-%m-%d %H:%M}")
print(f"   VAL   : {full_idx[i_tr]:%Y-%m-%d %H:%M} → {VAL_END:%Y-%m-%d %H:%M}")
print(f"   TEST  : {full_idx[i_va]:%Y-%m-%d %H:%M} → {full_idx[-1]:%Y-%m-%d %H:%M}")

def make_supervised(target, horizon):
    # Returns leakage-safe train/val/test splits for one (target, horizon).
    y = model_df[target].ffill(limit=3)            # short, past-only gap fill
    X = build_features(y)
    data = X.copy()
    data["__y"] = y.shift(-horizon).values          # y(t+h) aligned to origin t
    data = data.dropna()                            # drop lag warm-up, gaps, and tail
    tr = data.index <= TRAIN_END
    va = (data.index > TRAIN_END) & (data.index <= VAL_END)
    te = data.index > VAL_END
    pack = lambda m: (data.loc[m, FEATURE_COLS], data.loc[m, "__y"])
    Xtr, ytr = pack(tr); Xva, yva = pack(va); Xte, yte = pack(te)
    return {"Xtr": Xtr, "ytr": ytr, "Xva": Xva, "yva": yva, "Xte": Xte, "yte": yte,
            "target": target, "horizon": horizon}

def mase_scale(target):
    # In-sample one-step seasonal-naive MAE on TRAIN only (denominator for MASE).
    y = model_df[target].ffill(limit=3)
    ytr = y[y.index <= TRAIN_END].dropna().values
    if len(ytr) <= SEASONAL_M:
        return np.nan
    return float(np.mean(np.abs(ytr[SEASONAL_M:] - ytr[:-SEASONAL_M])))

MASE_SCALE = {t: mase_scale(t) for t in TARGETS + FLOOR_TARGETS}

# sanity: show supervised sizes for the primary target
for h in HORIZONS:
    s = make_supervised(PRIMARY_TARGET, h)
    print(f"   {PRIMARY_TARGET} h={h:>2}: train={len(s['ytr'])}, val={len(s['yva'])}, test={len(s['yte'])}")

# ---- visualize the split on the primary target ----
ys = model_df[PRIMARY_TARGET]
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(ys.index, ys.values, color=COLORS["light"], lw=0.6)
ax.axvspan(full_idx[0], TRAIN_END, color=COLORS["comfort"], alpha=0.10, label="train")
ax.axvspan(TRAIN_END, VAL_END, color=COLORS["accent2"], alpha=0.12, label="val")
ax.axvspan(VAL_END, full_idx[-1], color=COLORS["energy"], alpha=0.12, label="test")
ax.set_title(f"Chronological split — {PRIMARY_TARGET} (hourly)")
ax.set_ylabel("Power (kW)"); ax.legend(loc="upper right", framealpha=0.3)
plt.tight_layout(); plt.show()

## 7 · Metrics & baselines (the honesty layer)

No model is allowed to claim victory without beating **simple baselines** on the
**same test window**. We report a full metric suite per target & horizon:

- **MAE**, **RMSE** — absolute error in kW
- **sMAPE**, **MAPE** — scale-free % error
- **MASE** — error relative to an in-sample seasonal-naïve (m = 24); **MASE < 1 ⇒ better than seasonal-naïve**
- **Peak-error** — mean absolute error of the **daily peak** demand (kW), what a grid/operations team cares about

**Baselines:** (1) **Persistence** `ŷ(t+h)=y(t)`, (2) **Seasonal-naïve** `ŷ(t+h)=y(t+h−24)`.
**ML:** Ridge (val-tuned α), HistGradientBoosting, RandomForest, plus LightGBM/XGBoost *if installed*.

In [ ]:
# ============================================================
# METRICS
# ============================================================
from sklearn.metrics import mean_absolute_error
EPS = 1e-6

def _smape(y, p):
    return float(np.mean(2*np.abs(y-p) / (np.abs(y)+np.abs(p)+EPS)) * 100)

def _mape(y, p):
    m = np.abs(y) > 1.0      # ignore near-zero denominators (kW); avoids explosions
    return float(np.mean(np.abs((y[m]-p[m])/y[m])) * 100) if m.any() else np.nan

def _peak_error(y, p, target_times):
    # mean over test days of |daily-max actual - daily-max predicted| (kW)
    df = pd.DataFrame({"y": y, "p": p}, index=pd.DatetimeIndex(target_times))
    g = df.groupby(df.index.date).agg(ymax=("y","max"), pmax=("p","max"))
    return float(np.mean(np.abs(g["ymax"] - g["pmax"])))

def compute_metrics(y_true, y_pred, scale, target_times):
    y = np.asarray(y_true, float); p = np.asarray(y_pred, float)
    tt = pd.DatetimeIndex(target_times)
    mask = np.isfinite(y) & np.isfinite(p)
    y, p, tt = y[mask], p[mask], tt[mask]
    if len(y) == 0:
        return {k: np.nan for k in ["MAE","RMSE","sMAPE","MAPE","MASE","PeakErr","n"]}
    mae = float(np.mean(np.abs(y-p)))
    return {
        "MAE": mae,
        "RMSE": float(np.sqrt(np.mean((y-p)**2))),
        "sMAPE": _smape(y, p),
        "MAPE": _mape(y, p),
        "MASE": float(mae/scale) if scale and np.isfinite(scale) and scale > 0 else np.nan,
        "PeakErr": _peak_error(y, p, tt),
        "n": int(len(y)),
    }
print("✅ metric functions ready (MAE, RMSE, sMAPE, MAPE, MASE, PeakErr)")

In [ ]:
# ============================================================
# MODELS — baselines + ML (sklearn always; LightGBM/XGBoost optional)
# ============================================================
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ML_MODELS = ["Ridge", "HistGBM", "RandomForest"]
if HAS_LGBM: ML_MODELS.append("LightGBM")
if HAS_XGB:  ML_MODELS.append("XGBoost")
BASELINES = ["Persistence", "SeasonalNaive"]

def _yfull(target):
    return model_df[target].ffill(limit=3)

def persistence_pred(target, origins):
    return _yfull(target).reindex(origins).values                  # y(t)

def seasonal_naive_pred(target, origins, horizon, m=SEASONAL_M):
    src = (origins + pd.Timedelta(hours=horizon)) - pd.Timedelta(hours=m)
    return _yfull(target).reindex(src).values                      # y(t+h-m)

def fit_predict_ml(name, s):
    Xtr, ytr, Xva, yva, Xte = s["Xtr"], s["ytr"], s["Xva"], s["yva"], s["Xte"]
    if name == "Ridge":
        best = None
        for alpha in [0.1, 1.0, 10.0, 100.0]:                      # selected on VALIDATION
            mdl = make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(Xtr, ytr)
            v = mean_absolute_error(yva, mdl.predict(Xva))
            if best is None or v < best[0]:
                best = (v, mdl)
        return best[1].predict(Xte)
    if name == "HistGBM":
        mdl = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    elif name == "RandomForest":
        mdl = RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE)
    elif name == "LightGBM":
        import lightgbm as lgb
        mdl = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05,
                                random_state=RANDOM_STATE, verbosity=-1)
    elif name == "XGBoost":
        import xgboost as xgb
        mdl = xgb.XGBRegressor(n_estimators=400, learning_rate=0.05,
                               random_state=RANDOM_STATE, verbosity=0)
    else:
        raise ValueError(name)
    mdl.fit(Xtr, ytr)
    return mdl.predict(Xte)

def predict_model(name, s):
    origins, h, tgt = s["Xte"].index, s["horizon"], s["target"]
    if name == "Persistence":   return persistence_pred(tgt, origins)
    if name == "SeasonalNaive": return seasonal_naive_pred(tgt, origins, h)
    return fit_predict_ml(name, s)

def evaluate_model(name, s):
    t0 = time.time()
    yp = predict_model(name, s)
    rt = time.time() - t0
    tt = s["Xte"].index + pd.Timedelta(hours=s["horizon"])
    row = compute_metrics(s["yte"].values, yp, MASE_SCALE[s["target"]], tt)
    row.update({"model": name, "target": s["target"], "horizon": s["horizon"],
                "type": "baseline" if name in BASELINES else "ml", "runtime_s": round(rt, 3)})
    return row, yp

print(f"✅ models ready | baselines={BASELINES} | ml={ML_MODELS}")

In [ ]:
# ============================================================
# EVALUATE all models × targets × horizons on the SAME test window
# ============================================================
ALL_MODELS = BASELINES + ML_MODELS
rows, PRED_CACHE = [], {}
print("⏳ Training & evaluating (chronological, leakage-safe)...")
for tgt in TARGETS:
    for h in HORIZONS:
        s = make_supervised(tgt, h)
        for name in ALL_MODELS:
            row, yp = evaluate_model(name, s)
            rows.append(row)
            PRED_CACHE[(tgt, h, name)] = (s["Xte"].index, s["yte"].values, yp)
    print(f"   ✓ {tgt}")
results_df = pd.DataFrame(rows)

# mark the ML model selected on VALIDATION (for the day-ahead primary target)
def best_ml_by_val(target, horizon):
    s = make_supervised(target, horizon)
    scores = {}
    for name in ML_MODELS:
        yp_val = (fit_predict_ml(name, {**s, "Xte": s["Xva"]}))
        scores[name] = mean_absolute_error(s["yva"], yp_val)
    return min(scores, key=scores.get), scores

sel_primary, _ = best_ml_by_val(PRIMARY_TARGET, 1)
print(f"\n🏅 ML model selected on VALIDATION for {PRIMARY_TARGET} h=1: {sel_primary}")
print(f"   (test metrics are reported below — validation was used only for selection)")

In [ ]:
# ============================================================
# LEADERBOARD
# ============================================================
def leaderboard(target, horizon):
    d = results_df[(results_df.target == target) & (results_df.horizon == horizon)].copy()
    d = d.sort_values("MAE")
    cols = ["model", "type", "MAE", "RMSE", "sMAPE", "MASE", "PeakErr", "runtime_s"]
    return d[cols].reset_index(drop=True)

for h in HORIZONS:
    print(f"\n{'='*78}\n🏁 LEADERBOARD — {PRIMARY_TARGET}  |  horizon = {h}h  "
          f"(test: {VAL_END+pd.Timedelta(hours=1):%Y-%m-%d} → {full_idx[-1]:%Y-%m-%d})\n{'='*78}")
    lb = leaderboard(PRIMARY_TARGET, h)
    with pd.option_context("display.float_format", lambda v: f"{v:8.3f}"):
        print(lb.to_string(index=False))
    best = lb.iloc[0]
    sn = lb[lb.model == "SeasonalNaive"].iloc[0]
    verdict = ("beats" if best.MAE < sn.MAE else "does NOT beat")
    print(f"→ Best: {best.model} (MAE {best.MAE:.2f} kW, MASE {best.MASE:.3f}); "
          f"{verdict} seasonal-naïve (MAE {sn.MAE:.2f}).")

# Compact visual: MAE by model for the primary target, both horizons
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(15, 5), squeeze=False)
for j, h in enumerate(HORIZONS):
    lb = leaderboard(PRIMARY_TARGET, h)
    clr = [COLORS["energy"] if t == "ml" else COLORS["comfort"] for t in lb["type"]]
    ax = axes[0][j]
    ax.barh(lb["model"][::-1], lb["MAE"][::-1], color=clr[::-1], alpha=0.9)
    ax.set_title(f"{PRIMARY_TARGET} — h={h}h  (MAE, kW)"); ax.set_xlabel("MAE (kW)")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# DIAGNOSTICS — forecast vs actual, error by hour & weekday/weekend
# ============================================================
def best_model_name(target, horizon):
    return leaderboard(target, horizon).iloc[0]["model"]

H_DIAG = 1
bm = best_model_name(PRIMARY_TARGET, H_DIAG)
origins, yt, yp = PRED_CACHE[(PRIMARY_TARGET, H_DIAG, bm)]
_, _, yp_sn     = PRED_CACHE[(PRIMARY_TARGET, H_DIAG, "SeasonalNaive")]
tt = origins + pd.Timedelta(hours=H_DIAG)

err_df = pd.DataFrame({"abs_err": np.abs(yt - yp)}, index=pd.DatetimeIndex(tt))
err_df["hour"] = err_df.index.hour
err_df["is_weekend"] = (err_df.index.dayofweek >= 5).astype(int)
by_hour = err_df.groupby("hour")["abs_err"].mean()
by_part = err_df.groupby("is_weekend")["abs_err"].mean()

fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(2, 2, hspace=0.33, wspace=0.22, top=0.92, left=0.07, right=0.97)
fig.suptitle(f"Forecast diagnostics — {PRIMARY_TARGET}, best model = {bm} (h={H_DIAG}h)",
             fontsize=15, fontweight="bold")

ax1 = fig.add_subplot(gs[0, :])
sl = slice(0, min(24*7, len(yt)))    # first test week
ax1.plot(tt[sl], yt[sl], color="white", lw=1.6, label="Actual")
ax1.plot(tt[sl], yp[sl], color=COLORS["energy"], lw=1.4, label=f"{bm}")
ax1.plot(tt[sl], yp_sn[sl], color=COLORS["comfort"], lw=1.0, ls="--", alpha=0.8, label="Seasonal-naïve")
ax1.set_ylabel("Power (kW)"); ax1.set_title("Test-window forecast vs actual (first week)")
ax1.legend(framealpha=0.3); ax1.xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))

ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(by_hour.index, by_hour.values, color=COLORS["energy"], alpha=0.9)
ax2.set_xlabel("Hour of day (target time)"); ax2.set_ylabel("MAE (kW)")
ax2.set_title("Error by hour of day"); ax2.set_xticks(range(0, 24, 3))

ax3 = fig.add_subplot(gs[1, 1])
ax3.bar(["Weekday", "Weekend"], by_part.reindex([0, 1]).values,
        color=[COLORS["energy"], COLORS["comfort"]], alpha=0.9)
ax3.set_ylabel("MAE (kW)"); ax3.set_title("Error: weekday vs weekend")
plt.show()

print(f"MAE by hour — worst hours: "
      f"{list(by_hour.sort_values(ascending=False).head(3).round(1).items())}")
print(f"Weekday MAE {by_part.get(0, np.nan):.2f} kW | Weekend MAE {by_part.get(1, np.nan):.2f} kW")

### 7b · Load-type & floor-level forecasts

The same pipeline forecasts each **load type** (AC / lighting / plug) and each
**floor total** — these feed the optimizer (which needs an AC-load forecast) and
show where forecasting is easy vs hard. Floor-level is included **only where data
quality allows** (Floor 6 has a truncated record, so its test sample is smaller).

In [ ]:
# ============================================================
# Load-type leaderboard (h=1) + floor-level quick forecast (h=1)
# ============================================================
print("📋 Load-type test MAE/MASE @ h=1 (best model per target):")
for tgt in TARGETS:
    lb = leaderboard(tgt, 1).iloc[0]
    print(f"   {tgt:22s} best={lb.model:13s} MAE={lb.MAE:7.2f} kW  MASE={lb.MASE:5.3f}")

# Floor-level: evaluate HistGBM vs SeasonalNaive at h=1 (compact)
floor_rows = []
for tgt in FLOOR_TARGETS:
    s = make_supervised(tgt, 1)
    if len(s["yte"]) < 24:
        continue
    for name in ["SeasonalNaive", "HistGBM"]:
        row, _ = evaluate_model(name, s)
        floor_rows.append(row)
if floor_rows:
    fdf = pd.DataFrame(floor_rows)
    piv = fdf.pivot_table(index="target", columns="model", values="MAE").round(2)
    piv["best"] = piv.idxmin(axis=1)
    print("\n📋 Floor-level MAE (kW) @ h=1:")
    print(piv.to_string())

## 8 · Time-series foundation model (OPTIONAL, zero-shot)

We add a modern time-series **foundation model — Amazon Chronos-Bolt** — as a
**zero-shot** forecaster (no training on our data). It is **strictly optional**:

| Property | Value |
|---|---|
| Mode | **Zero-shot** (pretrained; no fine-tuning) |
| Input | **Univariate** load history (no covariates) |
| Context length | last **512** hours |
| Horizon | predicts `h` steps; we take the `h`-th |
| Hardware | CPU is fine for `*-small`; GPU optional |
| Requirement | `chronos-forecasting` + `torch` + model weights (internet **or** pre-attached) |

**Guard rails:** the default is `USE_FOUNDATION_MODEL = False`. If the package, weights,
internet or GPU are unavailable, this section **prints why and is skipped** — the
guaranteed baselines/ML above already cover the whole pipeline. Foundation results
are evaluated on the **same test origins** and compared to seasonal-naïve and the
best ML model on an **identical** (capped) origin subset, so the comparison is fair.

> **▶ Running it (default `USE_FOUNDATION_MODEL = True`):** the next cell **auto-installs**
> `chronos-forecasting` (it imports as `chronos`) and downloads `amazon/chronos-bolt-small`.
> - **Colab** — works out of the box (internet is on; `torch` is pre-installed).
> - **Kaggle** — first turn **Settings → Internet → On** (needs a phone-verified account); `torch` is pre-installed.
> - **No internet** — install/download fails and the section **skips cleanly** (that was the
>   `No module named 'chronos'` message). Set `USE_FOUNDATION_MODEL=False` to force baseline-only.

In [ ]:
# ============================================================
# FOUNDATION MODEL — Amazon Chronos-Bolt (zero-shot, guarded) — OPTIONAL
# ============================================================
foundation_results = []
FOUNDATION_STATUS = "skipped"
CONTEXT_LEN = 512
# The PyPI package is `chronos-forecasting` but it IMPORTS as `chronos`.
# On Kaggle: enable Internet (Settings → Internet → On) so it can pip-install AND
# download the model weights from Hugging Face. torch is pre-installed on Kaggle.
CHRONOS_PIP_SPEC  = "chronos-forecasting"
ALLOW_PIP_INSTALL = True      # auto-install chronos when USE_FOUNDATION_MODEL=True and it's missing

def _pip_install(spec):
    import subprocess, sys, importlib
    print(f"   • pip install {spec} (requires internet)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec],
                       capture_output=True, text=True)
    if r.returncode != 0:
        tail = (r.stderr or r.stdout or "pip failed").strip().splitlines()[-1:]
        raise RuntimeError(tail[0] if tail else "pip failed")
    importlib.invalidate_caches()

def _chronos_available():
    if not USE_FOUNDATION_MODEL:
        return False, "USE_FOUNDATION_MODEL=False (default, Kaggle/offline-safe)"
    # torch is required and is NOT auto-installed (large; pre-installed on Kaggle).
    try:
        import torch  # noqa: F401
    except Exception as e:
        return False, (f"PyTorch missing ({type(e).__name__}). On Kaggle torch is pre-installed; "
                       "locally run `pip install torch`.")
    try:
        import chronos  # noqa: F401
        return True, "ok"
    except ImportError:
        if not ALLOW_PIP_INSTALL:
            return False, f"`chronos` not installed (set ALLOW_PIP_INSTALL=True or `pip install {CHRONOS_PIP_SPEC}`)"
        try:
            _pip_install(CHRONOS_PIP_SPEC)
            import chronos  # noqa: F401
            return True, f"auto-installed {CHRONOS_PIP_SPEC}"
        except Exception as e:
            return False, (f"could not install {CHRONOS_PIP_SPEC} ({type(e).__name__}: {e}). "
                           "On Kaggle enable Internet, or attach the wheel/weights as a dataset.")

def _load_chronos():
    import torch
    from chronos import BaseChronosPipeline   # downloads weights on first call (needs internet)
    return BaseChronosPipeline.from_pretrained(
        CHRONOS_MODEL, device_map="cpu", torch_dtype=torch.float32)

def _chronos_batch_predict(pipeline, contexts, prediction_length):
    import torch
    ctx = [torch.tensor(np.asarray(c, dtype="float32")) for c in contexts]
    # API differs across versions; try quantiles first, fall back to samples
    try:
        q, _mean = pipeline.predict_quantiles(
            context=ctx, prediction_length=prediction_length, quantile_levels=[0.1, 0.5, 0.9])
        return np.asarray(q[..., 1])                       # median -> [N, h]
    except Exception:
        fc = pipeline.predict(ctx, prediction_length)      # [N, samples, h]
        return np.median(np.asarray(fc), axis=1)

def make_chronos_predict_fn(pipeline):
    def _fn(target, horizon, origins):
        y = _yfull(target)
        contexts = [y.loc[:t].dropna().values[-CONTEXT_LEN:] for t in origins]
        preds = _chronos_batch_predict(pipeline, contexts, horizon)  # [N, h]
        return np.asarray(preds)[:, horizon-1]
    return _fn

def evaluate_foundation(target, horizon, predict_fn, max_windows=CHRONOS_MAX_WINDOWS):
    # Compares the foundation model to seasonal-naïve + HistGBM on the SAME origins.
    s = make_supervised(target, horizon)
    origins = s["Xte"].index
    if len(origins) > max_windows:                          # cap to bound runtime (logged)
        sel = np.linspace(0, len(origins)-1, max_windows).astype(int)
        origins = origins[np.unique(sel)]
        print(f"   • capping to {len(origins)} evenly-spaced test origins (of {len(s['Xte'])})")
    tt = origins + pd.Timedelta(hours=horizon)
    yt = _yfull(target).reindex(tt).values
    scale = MASE_SCALE[target]
    def _row(name, typ, yp, rt):
        r = compute_metrics(yt, yp, scale, tt)
        r.update({"model": name, "type": typ, "target": target, "horizon": horizon,
                  "runtime_s": round(rt, 2), "n_windows": len(origins)})
        return r
    t0 = time.time(); yp_f = predict_fn(target, horizon, origins); rt = time.time()-t0
    rows = [_row("Chronos-Bolt(zs)", "foundation", yp_f, rt)]
    rows.append(_row("SeasonalNaive", "baseline", seasonal_naive_pred(target, origins, horizon), 0.0))
    sub = {**s, "Xte": s["Xte"].loc[origins]}
    t0 = time.time(); yp_ml = fit_predict_ml("HistGBM", sub); rt = time.time()-t0
    rows.append(_row("HistGBM", "ml", yp_ml, rt))
    return rows

ok, msg = _chronos_available()
if ok:
    try:
        print(f"⏳ Loading {CHRONOS_MODEL} (zero-shot)...")
        _pipe = _load_chronos()
        _fn = make_chronos_predict_fn(_pipe)
        for h in HORIZONS:
            print(f"   • evaluating Chronos zero-shot, {PRIMARY_TARGET}, h={h}")
            foundation_results += evaluate_foundation(PRIMARY_TARGET, h, _fn)
        FOUNDATION_STATUS = "ran"
        print("✅ Foundation-model evaluation complete.")
    except Exception as e:
        FOUNDATION_STATUS = f"failed ({type(e).__name__})"
        print(f"⚠️ Foundation model failed at runtime: {type(e).__name__}: {e}")
        print("   → Skipping; the guaranteed baselines/ML above already cover the pipeline.")
else:
    print(f"ℹ️ Foundation-model section SKIPPED — {msg}")
    print("   To enable: set USE_FOUNDATION_MODEL=True with `chronos-forecasting`+`torch`")
    print("   installed and model weights reachable (internet or pre-attached dataset).")
    print("   The notebook runs end-to-end without it.")

In [ ]:
# ============================================================
# FOUNDATION vs BASELINES — fair comparison on identical origins
# ============================================================
if foundation_results:
    fr = pd.DataFrame(foundation_results)
    cols = ["target", "horizon", "model", "type", "MAE", "RMSE", "sMAPE", "MASE", "PeakErr", "runtime_s"]
    for h in HORIZONS:
        sub = fr[fr.horizon == h].sort_values("MAE")
        print(f"\n🔬 Foundation comparison — {PRIMARY_TARGET}, h={h}h "
              f"(identical {int(sub['n_windows'].iloc[0])} test origins):")
        with pd.option_context("display.float_format", lambda v: f"{v:8.3f}"):
            print(sub[cols].to_string(index=False))
    best = fr.sort_values('MAE').iloc[0]
    print(f"\nEvidence-based verdict: lowest MAE overall = {best.model} "
          f"({best.MAE:.2f} kW at h={int(best.horizon)}).")
    print("We report what the metrics show — no automatic claim that the foundation model wins.")
else:
    print("No foundation-model results to compare (section was skipped or failed).")
    print("Conceptual note: foundation models are attractive for *zero-shot* use — no")
    print("per-series training — and for cold-start buildings with little history. On a")
    print("data-rich single building, a well-tuned gradient-boosting model with calendar")
    print("+ lag features is a very strong, cheap baseline and often competitive or better.")

## 9 · Anomaly detection — forecast-residual method vs Isolation Forest

The original notebook used **Isolation Forest** with a hand-fixed
`contamination=0.03` as its only model. We keep it as a **baseline** (with
`contamination='auto'`, removing the unvalidated constant) and add a more
**explainable, forecast-driven** detector:

$$\text{score}(t) = \frac{|\,y(t) - \hat{y}(t)\,|}{\sigma_{\text{resid}}^{\text{val}}}$$

where $\hat{y}$ is the one-step forecast and $\sigma_{\text{resid}}^{\text{val}}$ is
the residual std estimated on **validation** (never on test). A point is anomalous
when the model is **surprised** (`score > Z`). We also encode three **operational
rules** (off-hours load, AC running while already comfortable, unoccupied
light/plug load). **No labels exist**, so we report **anomaly rate, method overlap,
and the top-10 suspicious hours** — not precision/recall.

In [ ]:
# ============================================================
# ANOMALY DETECTION — forecast residuals + rules + Isolation Forest
# ============================================================
from sklearn.ensemble import IsolationForest

Z_THRESH = 3.5
s = make_supervised(PRIMARY_TARGET, 1)
_anom_mdl = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(s["Xtr"], s["ytr"])
val_pred, test_pred = _anom_mdl.predict(s["Xva"]), _anom_mdl.predict(s["Xte"])
sigma_resid = float((s["yva"].values - val_pred).std())     # expected error from VALIDATION

# held-out (val+test), indexed by TARGET time t+1
ho_origins = s["Xva"].index.append(s["Xte"].index)
ho_tt   = ho_origins + pd.Timedelta(hours=1)
actual  = np.r_[s["yva"].values, s["yte"].values]
pred    = np.r_[val_pred, test_pred]
A = pd.DataFrame(index=pd.DatetimeIndex(ho_tt))
A["actual"], A["pred"] = actual, pred
A["resid"] = A["actual"] - A["pred"]
A["score"] = (A["resid"].abs() / (sigma_resid + EPS))
for c in ["Bldg_AC_total_kW", "Bldg_Light_total_kW", "Bldg_Plug_total_kW", "Bldg_Temp_avg"]:
    A[c] = model_df[c].reindex(A.index).values
H = A.index.hour.to_numpy(); D = A.index.dayofweek.to_numpy()
A["is_working_hours"] = (((H >= 8) & (H < 18)) & (D < 5)).astype(int)

# thresholds learned on TRAIN only
tr = add_calendar(model_df[model_df.index <= TRAIN_END])
off_tr = tr[tr["is_working_hours"] == 0]
thr = {
    "off_load": float(off_tr["Bldg_Total_kW"].quantile(0.95)),
    "ac":       float(tr["Bldg_AC_total_kW"].quantile(0.75)),
    "light":    float(tr["Bldg_Light_total_kW"].quantile(0.75)),
    "plug":     float(tr["Bldg_Plug_total_kW"].quantile(0.90)),
}

A["anom_residual"] = A["score"] > Z_THRESH
A["rule_offhours_load"] = (A["is_working_hours"] == 0) & (A["actual"] > thr["off_load"])
A["rule_ac_while_comfortable"] = ((A["Bldg_AC_total_kW"] > thr["ac"]) &
                                  A["Bldg_Temp_avg"].notna() &
                                  (A["Bldg_Temp_avg"] <= COMFORT["temp_high"]))
A["rule_unoccupied_plugload"] = ((A["is_working_hours"] == 0) &
                                 ((A["Bldg_Light_total_kW"] > thr["light"]) |
                                  (A["Bldg_Plug_total_kW"] > thr["plug"])))
A["anom_rule_any"] = A[["rule_offhours_load","rule_ac_while_comfortable","rule_unoccupied_plugload"]].any(axis=1)

# Isolation Forest baseline (contamination='auto' — no hand-fixed constant)
if_feat = pd.DataFrame(index=A.index)
if_feat["load"] = A["actual"]
if_feat["roll4h"] = model_df[PRIMARY_TARGET].rolling(4, min_periods=1).mean().reindex(A.index).values
if_feat["diff1"] = model_df[PRIMARY_TARGET].diff().reindex(A.index).values
if_feat["hour"] = H
if_feat = if_feat.fillna(if_feat.median())
_if = IsolationForest(contamination="auto", random_state=RANDOM_STATE, n_estimators=200)
A["anom_iforest"] = (_if.fit_predict(StandardScaler().fit_transform(if_feat)) == -1)

# ---- compare ----
res_set, if_set = set(A.index[A["anom_residual"]]), set(A.index[A["anom_iforest"]])
inter, uni = len(res_set & if_set), len(res_set | if_set)
print("🔎 ANOMALY DETECTION (held-out val+test, no labels)")
print("="*64)
print(f"   Residual method  : {A['anom_residual'].sum():4d} flags "
      f"({A['anom_residual'].mean()*100:.2f}%)  σ_resid(val)={sigma_resid:.2f} kW, Z={Z_THRESH}")
print(f"   Rule-based (any) : {A['anom_rule_any'].sum():4d} flags ({A['anom_rule_any'].mean()*100:.2f}%)")
print(f"   IsolationForest  : {A['anom_iforest'].sum():4d} flags ({A['anom_iforest'].mean()*100:.2f}%)")
print(f"   Residual∩IForest : {inter}  (Jaccard={inter/max(uni,1):.2f})")
print("\n   Top-10 most surprising hours (forecast-residual score):")
top = A.sort_values("score", ascending=False).head(10)
for ts, r in top.iterrows():
    tags = [t for t, on in [("offhrs", r.rule_offhours_load),
            ("ac_waste", r.rule_ac_while_comfortable), ("unocc", r.rule_unoccupied_plugload)] if on]
    print(f"     {ts:%Y-%m-%d %H:%M}  actual={r.actual:6.1f} pred={r.pred:6.1f} "
          f"z={r.score:4.1f}  {','.join(tags) if tags else '-'}")

In [ ]:
# ---- visualize anomalies on the held-out window ----
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(A.index, A["actual"], color="white", lw=0.8, label="Actual")
axes[0].plot(A.index, A["pred"], color=COLORS["comfort"], lw=0.8, alpha=0.8, label="1h forecast")
rr = A[A["anom_residual"]]
axes[0].scatter(rr.index, rr["actual"], color=COLORS["bad"], s=22, zorder=5,
                label=f"Residual anomaly ({len(rr)})")
axes[0].axvline(VAL_END, color=COLORS["accent2"], ls="--", alpha=0.6)
axes[0].set_ylabel("Power (kW)"); axes[0].legend(framealpha=0.3, ncol=4, fontsize=9)
axes[0].set_title(f"{PRIMARY_TARGET}: forecast vs actual with residual anomalies "
                  f"(dashed = val/test boundary)")
axes[1].fill_between(A.index, 0, A["score"], color=COLORS["energy"], alpha=0.5)
axes[1].axhline(Z_THRESH, color=COLORS["bad"], ls="--", label=f"Z={Z_THRESH}")
axes[1].set_ylabel("|resid| / σ"); axes[1].legend(framealpha=0.3, fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.tight_layout(); plt.show()
print("Explainability: the residual score says *how surprising* each hour is and the")
print("rules say *why* (off-hours load / cooling while comfortable / unoccupied plug load) —")
print("more actionable than an unlabeled Isolation-Forest flag.")

## 10 · Forecast-informed scheduling (replacing the arbitrary "AC → 10%" rule)

The original "smart schedule" cut off-hours AC to **10%** and pre-cooled at
**120%** — arbitrary constants that silently assume comfort is fine. We replace
this with two transparent, comfort-aware policies and account energy **correctly
in kWh**:

1. **Rule-based baseline** — trim AC **only** in hours that were **already
   unoccupied** (off-hours / weekend) **and** already within the comfort band, so
   occupant-facing comfort is unchanged *by construction*. (We have no thermal model,
   so we do **not** claim deeper cuts are safe — that risk is left explicit.)
2. **Forecast-informed peak-shaving** — use the **day-ahead AC forecast** to decide
   *which days* to act, then **cap-and-fill**: cap the day's AC at `(1−SHAVE)×peak`
   and **bank** the shaved energy by pre-cooling in the **low early-morning ramp**.
   Cap-and-fill **guarantees the daily peak never increases** and is **energy-neutral**
   apart from a small pre-cooling penalty — it cuts the **peak kW** that drives demand charges.

We report **kWh saved, peak-kW reduction, hours affected, ฿ and CO₂** — all from the
single `series_energy_kwh` helper and clearly-stated assumptions.

In [ ]:
# ============================================================
# OPTIMIZATION — rule-based trim vs forecast-informed peak-shaving
# ============================================================
opt = add_calendar(model_df)                      # hourly, with calendar
win = opt[opt.index > VAL_END].copy()             # evaluate on the held-out test window
ac = win["Bldg_AC_total_kW"].astype(float)

# ---- assumptions (explicit) ----
REDUCE_FRAC      = 0.40   # off-hours AC setpoint relaxation (modest, assumption)
SHAVE_FRAC       = 0.15   # target peak-shave: cap the day's AC peak at (1-SHAVE_FRAC)*peak
PRECOOL_PENALTY  = 0.05   # extra energy for pre-cooling (thermal losses, assumption)
PRECOOL_HOURS    = (5, 9) # bank coolth in the low early-morning ramp (inclusive)

# ---- (1) rule-based off-hours trim (comfort-neutral by construction) ----
eligible = (win["is_working_hours"] == 0) & (win["Bldg_Temp_avg"].fillna(0) <= COMFORT["temp_high"])
ac_rule = ac.copy()
ac_rule[eligible] = ac[eligible] * (1 - REDUCE_FRAC)
kwh_base  = series_energy_kwh(ac,       INTERVAL_H)
kwh_rule  = series_energy_kwh(ac_rule,  INTERVAL_H)
saved_rule = kwh_base - kwh_rule

# ---- (2) forecast-informed peak-shaving (cap-and-fill pre-cooling) ----
# Decision (which days to act) is driven by the DAY-AHEAD AC forecast; the energy
# we shave off the peak is *banked* earlier by pre-cooling in the low-load morning
# ramp. Cap-and-fill guarantees the daily peak never increases and energy is
# conserved apart from the small pre-cooling penalty.
s_ac = make_supervised("Bldg_AC_total_kW", 24)
_ac_mdl = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(s_ac["Xtr"], s_ac["ytr"])
ac_fc = pd.Series(_ac_mdl.predict(s_ac["Xte"]),
                  index=s_ac["Xte"].index + pd.Timedelta(hours=24)).reindex(win.index)
daily_pred_peak = ac_fc.groupby(ac_fc.index.date).max().dropna()
peak_thr = float(daily_pred_peak.median()) if len(daily_pred_peak) else np.inf  # act on above-median days

ac_shaved = ac.copy()
peak_before, peak_after, days_acted = [], [], 0
for day, g in win.groupby(win.index.date):
    fc_day = ac_fc.reindex(g.index)
    if fc_day.notna().sum() < 6 or fc_day.max() < peak_thr:
        continue                                       # forecast: not a high-peak day → skip
    load = ac.reindex(g.index).astype(float)
    P = float(load.max()); cap = P * (1 - SHAVE_FRAC)
    over = (load - cap).clip(lower=0)                  # energy above the cap (kW @ 1h = kWh)
    E_shave = float(over.sum())
    ramp = load[(load.index.hour >= PRECOOL_HOURS[0]) & (load.index.hour <= PRECOOL_HOURS[1])]
    headroom = (cap - ramp).clip(lower=0)              # room to pre-cool without making a new peak
    if E_shave <= 0 or headroom.sum() <= 0:
        continue
    ratio = min(1.0, float(headroom.sum()) / E_shave)  # only shave what we can bank
    new_day = load.copy()
    new_day.loc[over.index]     = new_day.loc[over.index] - over * ratio          # cap the peaks
    fill = headroom * (E_shave * ratio / float(headroom.sum()))
    new_day.loc[fill.index]     = new_day.loc[fill.index] + fill * (1 + PRECOOL_PENALTY)  # bank early
    ac_shaved.loc[new_day.index] = new_day.values
    peak_before.append(P); peak_after.append(float(new_day.max())); days_acted += 1

kwh_shaved = series_energy_kwh(ac_shaved, INTERVAL_H)
peak_red = float(np.mean(np.array(peak_before) - np.array(peak_after))) if days_acted else 0.0

# ---- report ----
def fmt_money(kwh): return f"฿{energy_cost_thb(kwh):,.0f}"
summary = pd.DataFrame([
    {"method": "Rule-based off-hours trim", "kWh_saved": round(saved_rule, 1),
     "pct_AC": round(saved_rule/max(kwh_base,EPS)*100, 1), "peak_kW_red": 0.0,
     "hours_affected": int(eligible.sum()),
     "comfort_risk": "low (only already-unoccupied & comfortable hours; temp-rise un-modeled)"},
    {"method": "Forecast-informed peak-shave", "kWh_saved": round(kwh_base-kwh_shaved, 1),
     "pct_AC": round((kwh_base-kwh_shaved)/max(kwh_base,EPS)*100, 1), "peak_kW_red": round(peak_red, 1),
     "hours_affected": days_acted,
     "comfort_risk": "low (pre-cool while occupied; energy ~neutral by design)"},
])
print("🛠  SCHEDULING OPTIONS (test window, energy via energy_kWh = kW × 1h)")
print("="*78)
print(f"   Baseline AC energy (test window): {kwh_base:,.0f} kWh   peak-threshold {peak_thr:.0f} kW")
print(summary.to_string(index=False))
print(f"\n   Rule-based saves {saved_rule:,.0f} kWh ≈ {fmt_money(saved_rule)} and "
      f"{energy_co2_kg(saved_rule):,.0f} kgCO2 over this window (assumptions above).")
print(f"   Peak-shave cuts mean daily AC peak by {peak_red:.1f} kW on {days_acted} acted days "
      f"(reduces demand charges; ~energy-neutral).")
print("   NOTE: figures are ILLUSTRATIVE what-ifs, not validated by a thermal/occupancy model.")

In [ ]:
# ---- visualize one representative test week ----
wk = win.loc[win.index[:24*7]]
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(wk.index, ac.reindex(wk.index), color=COLORS["energy"], lw=1.4, label="Observed AC")
ax.plot(wk.index, ac_rule.reindex(wk.index), color=COLORS["maintain"], lw=1.2, label="Rule-based trim")
ax.plot(wk.index, ac_shaved.reindex(wk.index), color=COLORS["comfort"], lw=1.0, ls="--",
        label="Forecast peak-shave")
for d in pd.date_range(wk.index[0].normalize(), wk.index[-1].normalize()):
    ax.axvspan(d + pd.Timedelta(hours=8), d + pd.Timedelta(hours=18), color="white", alpha=0.04)
ax.set_ylabel("AC power (kW)"); ax.set_title("Scheduling policies — one test week (shaded = working hours)")
ax.legend(framealpha=0.3, ncol=3); ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.tight_layout(); plt.show()

## 11 · A *real* multi-agent architecture (classes, responsibilities, coordination)

The original notebook described agents in prose. Here each agent is an actual
**class** with a clear responsibility, operating on the objects computed above:

| Agent | Responsibility | Consumes |
|---|---|---|
| `DataQualityAgent` | flag unreliable channels | `data_quality_report` |
| `ForecastingAgent` | forecasts + prediction interval + confidence | models, `results_df` |
| `ComfortAgent` | comfort violations; **veto** unsafe actions | `comfort_df`, `COMFORT` |
| `AnomalyAgent` | surface & explain abnormal hours | residual frame `A` |
| `OptimizationAgent` | propose candidate savings actions | optimizer outputs |
| `CoordinatorAgent` | vet (reject comfort-violating), score, **rank** | all of the above |

The **Coordinator** is where coordination actually happens: it asks the
ComfortAgent to **veto** any action that would push an already-hot zone past the
comfort band, attaches a **confidence** from forecast accuracy and data quality,
and **ranks** the survivors by savings × confidence × (1 − risk).

In [ ]:
# ============================================================
# MULTI-AGENT ARCHITECTURE — explicit classes & coordination logic
# ============================================================
class DataQualityAgent:
    def __init__(self, report, quality_df, trust_missing_pct=10.0):
        self.report, self.quality_df, self.thr = report, quality_df, trust_missing_pct
    def audit(self):
        return {t: {"reliable": rec["avg_missing_pct"] <= self.thr, **rec}
                for t, rec in self.report.items()}
    def unreliable_channels(self):
        return self.quality_df[self.quality_df["missing_pct"] > self.thr]["column"].tolist()
    def reliability(self, target_kind="ac"):
        a = self.audit().get(target_kind, {})
        return float(max(0.0, 1.0 - a.get("avg_missing_pct", 50.0) / 100.0))

class ForecastingAgent:
    def __init__(self, results_df, predict_fn, make_supervised, z=1.28):
        self.results_df, self.predict_fn, self.make_supervised, self.z = results_df, predict_fn, make_supervised, z
    def skill(self, target, horizon):                      # test MASE of the best model
        d = self.results_df[(self.results_df.target == target) & (self.results_df.horizon == horizon)]
        return float(d["MASE"].min()) if len(d) else np.nan
    def confidence(self, target, horizon):                 # MASE→[0,1]; <1 beats seasonal-naïve
        m = self.skill(target, horizon)
        return float(np.clip(1.0 - m, 0.0, 1.0)) if np.isfinite(m) else 0.3
    def forecast(self, target, horizon, model="HistGBM"):
        s = self.make_supervised(target, horizon)
        yhat = self.predict_fn(model, s)
        sigma = float((s["yva"].values - self.predict_fn(model, {**s, "Xte": s["Xva"]})).std())
        tt = s["Xte"].index + pd.Timedelta(hours=horizon)
        return pd.DataFrame({"yhat": yhat, "lo": yhat - self.z*sigma, "hi": yhat + self.z*sigma},
                            index=tt), sigma

class ComfortAgent:
    def __init__(self, comfort_df, comfort_bands, safety_margin=0.5):
        self.comfort_df, self.C, self.margin = comfort_df, comfort_bands, safety_margin
    def summary(self):
        if self.comfort_df.empty:
            return {"mean_violation_pct": np.nan, "worst": None}
        w = self.comfort_df.nlargest(1, "violation_pct").iloc[0]
        return {"mean_violation_pct": float(self.comfort_df["violation_pct"].mean()),
                "worst_zone": w["label"], "worst_pct": float(w["violation_pct"]),
                "worst_mean_temp": float(w["mean_temp"])}
    def veto(self, action):
        # Reject AC-reducing actions where the affected zone is already near/over the upper band.
        t = action.get("temp_context", None)
        if action.get("reduces_cooling") and t is not None and t > self.C["temp_high"] - self.margin:
            return True, f"zone already at {t:.1f}°C (≥ {self.C['temp_high']-self.margin:.1f}°C) — cutting AC risks comfort"
        return False, "within comfort headroom"

class AnomalyAgent:
    def __init__(self, A):
        self.A = A
    def rate(self):
        return float(self.A["anom_residual"].mean() * 100)
    def top(self, k=5):
        cols = ["actual", "pred", "score", "rule_offhours_load",
                "rule_ac_while_comfortable", "rule_unoccupied_plugload"]
        return self.A.sort_values("score", ascending=False).head(k)[cols]

class OptimizationAgent:
    def __init__(self, rule_kwh, shave_kwh, peak_red, days_acted, hottest_zone, hottest_temp, hot_zone_kwh_est):
        self.rule_kwh, self.shave_kwh = rule_kwh, shave_kwh
        self.peak_red, self.days_acted = peak_red, days_acted
        self.hottest_zone, self.hottest_temp, self.hot_zone_kwh_est = hottest_zone, hottest_temp, hot_zone_kwh_est
    def propose(self):
        return [
            {"name": "Relax AC setpoint during off-hours",
             "kwh": self.rule_kwh, "peak": 0.0, "reduces_cooling": True,
             "temp_context": None,  # only already-unoccupied & comfortable hours → no veto
             "basis": "rule", "action": "BMS off-hours/weekend AC setpoint +2°C where unoccupied"},
            {"name": "Day-ahead pre-cool / peak-shave",
             "kwh": self.shave_kwh, "peak": self.peak_red, "reduces_cooling": False,
             "temp_context": None, "basis": "forecast(24h)",
             "action": f"On forecast high-peak days, pre-cool 05–09h; cap peak (acted {self.days_acted} days)"},
            {"name": f"Deep AC cut in hottest zone {self.hottest_zone}",
             "kwh": self.hot_zone_kwh_est, "peak": 0.0, "reduces_cooling": True,
             "temp_context": self.hottest_temp, "basis": "naive",
             "action": f"Cut AC in {self.hottest_zone} (UNVETTED proposal)"},
        ]

class CoordinatorAgent:
    def __init__(self, dqa, fa, ca, aa, oa):
        self.dqa, self.fa, self.ca, self.aa, self.oa = dqa, fa, ca, aa, oa
    def coordinate(self, target="Bldg_AC_total_kW", horizon=24):
        base_conf = self.fa.confidence(target, horizon)            # forecast skill
        rel = self.dqa.reliability("ac")                           # data reliability for AC
        rows = []
        for a in self.oa.propose():
            vetoed, why = self.ca.veto(a)
            # confidence blends forecast skill (for forecast actions) and data reliability
            conf = rel if a["basis"] == "rule" else (0.5*base_conf + 0.5*rel)
            risk = 0.6 if vetoed else (0.1 if a["basis"] != "naive" else 0.3)
            kwh = a["kwh"]
            score = 0.0 if vetoed else (abs(kwh if np.isfinite(kwh) else 0) + 5*a["peak"]) * conf * (1 - risk)
            rows.append({
                "Recommendation": a["name"],
                "Expected kWh saved": (round(kwh, 1) if np.isfinite(kwh) else "flag"),
                "Peak reduction (kW)": round(a["peak"], 1),
                "Comfort risk": ("REJECTED — " + why) if vetoed else "low",
                "Confidence": f"{conf:.2f} ({'High' if conf>0.66 else 'Med' if conf>0.4 else 'Low'})",
                "Required action": a["action"],
                "_score": round(score, 1), "_status": "rejected" if vetoed else "accepted",
            })
        # add an investigation item from the AnomalyAgent
        top = self.aa.top(1)
        if len(top):
            ts = top.index[0]
            rows.append({
                "Recommendation": "Investigate recurring anomaly cluster",
                "Expected kWh saved": "flag", "Peak reduction (kW)": 0.0,
                "Comfort risk": "low",
                "Confidence": f"{0.5:.2f} (Med)",
                "Required action": f"Audit AC/controls around {ts:%Y-%m-%d %H:%M} "
                                   f"(residual z={top.iloc[0]['score']:.1f})",
                "_score": 1.0, "_status": "accepted"})
        out = pd.DataFrame(rows).sort_values(["_status", "_score"], ascending=[True, False])
        return out

print("✅ Agent classes defined: DataQuality, Forecasting, Comfort, Anomaly, Optimization, Coordinator")

In [ ]:
# ============================================================
# RUN THE AGENTS → final ranked recommendation table
# ============================================================
# hottest zone (for the deliberately-risky proposal the Coordinator must veto)
_worst = comfort_df.nlargest(1, "violation_pct").iloc[0] if not comfort_df.empty else None
hottest_zone = _worst["label"] if _worst is not None else "n/a"
hottest_temp = float(_worst["mean_temp"]) if _worst is not None else 0.0
hot_zone_kwh_est = round(0.10 * kwh_base, 1)

dqa = DataQualityAgent(data_quality_report, quality_df)
fa  = ForecastingAgent(results_df, predict_model, make_supervised)
ca  = ComfortAgent(comfort_df, COMFORT)
aa  = AnomalyAgent(A)
oa  = OptimizationAgent(saved_rule, kwh_base - kwh_shaved, peak_red, days_acted,
                        hottest_zone, hottest_temp, hot_zone_kwh_est)
coord = CoordinatorAgent(dqa, fa, ca, aa, oa)

print("🧠 AGENT STATUS REPORT")
print("="*72)
qa = dqa.audit()
print("DataQualityAgent  : " + ", ".join(
    f"{k}={'OK' if v['reliable'] else 'LOW'}({v['avg_missing_pct']:.0f}%)" for k, v in qa.items()))
print(f"ForecastingAgent  : {PRIMARY_TARGET} h=1 MASE={fa.skill(PRIMARY_TARGET,1):.3f} "
      f"(conf {fa.confidence(PRIMARY_TARGET,1):.2f}); AC h=24 conf {fa.confidence('Bldg_AC_total_kW',24):.2f}")
cs = ca.summary()
print(f"ComfortAgent      : mean zone violation {cs['mean_violation_pct']:.1f}% | "
      f"worst {cs.get('worst_zone')} ({cs.get('worst_pct',0):.0f}%, {cs.get('worst_mean_temp',0):.1f}°C)")
print(f"AnomalyAgent      : residual anomaly rate {aa.rate():.2f}% of held-out hours")
print(f"OptimizationAgent : {len(oa.propose())} candidate actions proposed")

final = coord.coordinate()
print("\n📋 COORDINATOR — FINAL RANKED RECOMMENDATIONS")
print("="*72)
show = final.drop(columns=["_score"]).reset_index(drop=True)
with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(show.to_string(index=False))
n_rej = (final["_status"] == "rejected").sum()
print(f"\n→ Coordinator accepted {len(final)-n_rej} action(s) and REJECTED {n_rej} "
      f"(comfort veto). This is the coordination logic the original notebook only described.")

## 12 · Master leaderboard & evidence-based verdict

One consolidated table across **every model × target × horizon**, on the **same
held-out test window**. The headline summary below is computed **from the metrics**
— we let the numbers decide which model wins, and whether ML/foundation models
actually beat the simple baselines.

In [ ]:
# ============================================================
# MASTER LEADERBOARD (all models × targets × horizons) + headline verdict
# ============================================================
master = results_df.copy()
if foundation_results:
    master = pd.concat([master, pd.DataFrame(foundation_results)], ignore_index=True)
master["Notes"] = np.where(master["type"] == "baseline", "no training",
                    np.where(master["type"] == "foundation", "zero-shot", "feature-based"))
cols = ["target", "horizon", "model", "type", "MAE", "RMSE", "sMAPE", "MASE", "PeakErr", "runtime_s", "Notes"]
master = master[cols].sort_values(["target", "horizon", "MAE"]).reset_index(drop=True)
with pd.option_context("display.max_rows", 200, "display.float_format", lambda v: f"{v:8.3f}"):
    print(master.to_string(index=False))

print("\n" + "="*72 + "\n🧾 HEADLINE RESULTS (evidence-based, from test metrics)\n" + "="*72)
ml_wins = 0; total = 0
for tgt in TARGETS:
    for h in HORIZONS:
        lb = leaderboard(tgt, h); best = lb.iloc[0]
        sn = lb[lb.model == "SeasonalNaive"].iloc[0]
        ml_best = lb[lb.type == "ml"].sort_values("MAE").iloc[0]
        beat = ml_best.MAE < sn.MAE
        ml_wins += int(beat); total += 1
        print(f"  {tgt:20s} h={h:>2}: best={best.model:13s} MAE={best.MAE:6.2f} "
              f"MASE={best.MASE:5.3f} | best-ML {'beats' if beat else 'TIES/LOSES vs'} seasonal-naïve")
print(f"\n  ML beat the seasonal-naïve baseline in {ml_wins}/{total} (target,horizon) cases.")
print("  Verdict is data-driven: where a baseline wins, we keep the baseline. No overclaiming.")
try:
    master.to_csv("leaderboard.csv", index=False); print("\n  (saved leaderboard.csv)")
except Exception:
    pass

## 13 · Limitations (read before quoting any number)

- **One building, not a campus.** CU-BEMS is a **single** 7-floor Chulalongkorn
  building. Figures here **do not** represent all of Chulalongkorn University; any
  campus-scale number is a **linear illustration**, not a validated estimate.
- **2018–2019 data.** Operations, occupancy and equipment have likely changed;
  results may not reflect the building today.
- **Occupancy is not measured.** We use **calendar/working-hours proxies**; there is
  no ground-truth occupancy, badge or timetable feed.
- **Weather / academic calendar are optional and absent here.** The guaranteed
  pipeline uses **no external files**. Adding Bangkok weather, Thai public holidays
  and the university calendar would likely improve accuracy — provided as optional hooks.
- **Foundation models may not win.** On a data-rich single building, gradient-boosted
  trees with lag/calendar features are a very strong, cheap baseline; the foundation
  model's value is mainly **zero-shot / cold-start**, not guaranteed accuracy.
- **Savings ≠ deployment.** The scheduler is a **what-if** with explicit assumptions
  (tariff, CO₂ factor, pre-cooling penalty). There is **no thermal/occupancy model**,
  so comfort outcomes of deeper cuts are **not** guaranteed.
- **Comfort bands are approximate.** ASHRAE-55-style cooling-season office bands for a
  Bangkok context — **not** occupant-validated or safety-critical clinical limits.
- **kWh figures depend on the unit assumption** that channels are mean power (kW) per
  interval (the CU-BEMS convention); all conversions use `energy_kWh = kW × interval_h`.

## 14 · Future work — toward a Chulalongkorn smart-campus deployment

1. **Optional external covariates (already stubbed):** Bangkok weather (TMD), Thai
   public holidays, the **academic calendar** (semester / exam / break) and campus
   events — auto-skipped today, wired to drop in without breaking offline runs.
2. **Probabilistic forecasting:** quantile/conformal intervals for risk-aware
   scheduling and demand-charge management.
3. **Foundation-model fine-tuning:** light fine-tuning of Chronos/TimesFM on campus
   history; compare zero-shot vs fine-tuned vs gradient-boosting under equal budgets.
4. **Physics-informed comfort:** a grey-box thermal model so the OptimizationAgent can
   *prove* comfort is maintained before recommending deeper setpoint changes.
5. **Multi-building scale-out:** per-building agents with a campus CoordinatorAgent for
   load-shifting and aggregate peak management across faculties.
6. **Closed-loop control:** from recommendations to BMS set-points with human-in-the-loop
   approval, A/B evaluation, and measured (not simulated) savings.

---
### Reproducibility & deliverables map
Problem (§0) · Dataset & loading (§1) · Quality audit (§2) · Units kW→kWh (§3–4) ·
Comfort baseline (§5) · Features & chronological split (§6) · Metrics & baselines (§7) ·
Load-type/floor forecasts (§7b) · Foundation model, optional (§8) · Anomaly detection (§9) ·
Forecast-informed scheduling (§10) · Multi-agent architecture (§11) · Master leaderboard (§12) ·
Limitations (§13) · Future work (§14).

> **Runs offline on Kaggle with only the CU-BEMS dataset attached.** Foundation models,
> weather, holidays and the academic calendar are optional and skip cleanly when absent.